# PERSISTENT GOOGLE DRIVE CHECKPOINTING

This Colab is configured for long training runs. **Do not store the only copy of checkpoints under `/content`.**

Checkpoints are written after every completed epoch to:
`/content/drive/MyDrive/AMPORA-Hybrid-ANC/checkpoints/`

If the Colab runtime disconnects, reconnect, rerun the setup cells, mount Drive, rebuild the model/optimizer/scheduler, and load the latest checkpoint. Training resumes from the next epoch.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path('/content/AMPORA-Hybrid-ANC')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.colab_drive_checkpoint import mount_ampora_drive, save_checkpoint, load_latest_checkpoint
DRIVE_ROOT = mount_ampora_drive()
print('Persistent AMPORA root:', DRIVE_ROOT)


# AMPORA Hybrid ANC — Complete Google Colab Workflow

This notebook is the **software/ML completion path** for the existing AMPORA repository. It deliberately does not depend on FPGA/Raspberry Pi hardware. It covers dataset preparation, controlled mixing, leakage-resistant splits, DC-CRN training, classical baselines, reference-assisted low-band NLMS + high-band DC-CRN hybrid evaluation, unseen-noise testing, FP32/INT8 verification, latency benchmarking, plots, and a final evidence report.

**Important:** no metric is fabricated. A result is reported only after the corresponding cell is actually run. The low-band NLMS experiment is explicitly a **software reference-assisted simulation**: the injected noise waveform is used as the reference signal. This demonstrates the algorithmic hybrid path without claiming a physical microphone implementation.

## 0. Repository setup

In [ ]:
# If you opened this notebook from the GitHub repository in Colab, run this cell from the repo root.
# Otherwise clone your repository first and cd into it.
import os, sys, subprocess, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT/'src').exists():
    print('Current directory:', ROOT)
    print('Clone/open AMPORA-Hybrid-ANC and set ROOT to that repository before continuing.')
else:
    print('AMPORA root:', ROOT)


In [ ]:
# Install dependencies. Restart the runtime only if Colab asks you to.
!pip -q install -r requirements.txt


## 1. Configure data paths

In [ ]:
# Put public clean speech and categorized noise under Google Drive or Colab storage.
# Expected layout:
# data/raw/speech/*.wav
# data/raw/stationary/*.wav
# data/raw/nonstationary/*.wav
# data/raw/transient/*.wav
#
# For a real run, edit these paths. Keep raw datasets OUT of GitHub.
SPEECH_DIR = 'data/raw/speech'
NOISE_DIRS = ['data/raw/stationary','data/raw/nonstationary','data/raw/transient']
PROCESSED = 'data/processed'


## 2. Run automated software checks first

In [ ]:
!python scripts/verify_project.py


## 3. Prepare leakage-resistant manifests

In [ ]:
!python scripts/prepare_dataset.py \
  --speech "$SPEECH_DIR" \
  --noise "${NOISE_DIRS[0]}" "${NOISE_DIRS[1]}" "${NOISE_DIRS[2]}" \
  --out "$PROCESSED" \
  --seed 42


### Check the manifests

In [ ]:
import pandas as pd, pathlib
for f in ['train_speech.csv','val_speech.csv','test_speech.csv','noise.csv']:
    p=pathlib.Path(PROCESSED)/f
    print(f, 'exists=', p.exists(), 'rows=', len(pd.read_csv(p)) if p.exists() else 'NA')


## 4. Generate controlled test pairs

The same clean/noise pair is evaluated at configurable SNR values. The generated noise waveform is saved as `noise_reference`, which is used only for the reference-assisted NLMS baseline/hybrid experiment.

In [ ]:
!python scripts/prepare_demo_manifest.py \
  --speech-csv "$PROCESSED/test_speech.csv" \
  --noise-csv "$PROCESSED/noise.csv" \
  --out "$PROCESSED/eval_pairs" \
  --snrs -5,0,5,10,15 \
  --per-speech 1 \
  --seed 123


## 5. Train the DC-CRN

The current AMPORA architecture is retained: 16-kHz input, high-band processing, complex residual mask, base channels 16, and a unidirectional GRU. The training script uses SI-SDR plus a small log-magnitude spectral term. Start with a smoke run before a longer run.

In [ ]:
# Smoke run: proves the training pipeline works. It is NOT a final result.
!python scripts/train.py \
  --train-speech "$PROCESSED/train_speech.csv" \
  --val-speech "$PROCESSED/val_speech.csv" \
  --noise "$PROCESSED/noise.csv" \
  --epochs 1 \
  --batch-size 4 \
  --out models/colab_smoke


In [ ]:
# Final practical run. Increase epochs/data size only after the smoke test succeeds.
# !python scripts/train.py \
#   --train-speech "$PROCESSED/train_speech.csv" \
#   --val-speech "$PROCESSED/val_speech.csv" \
#   --noise "$PROCESSED/noise.csv" \
#   --epochs 30 \
#   --batch-size 8 \
#   --out models/colab_final


## 6. Evaluate the actual model + hybrid software path

In [ ]:
# Set this after training.
CHECKPOINT = 'models/colab_final/best.pt'


In [ ]:
# Run only after CHECKPOINT exists.
# !python scripts/evaluate_full_pipeline.py \
#   "$PROCESSED/eval_pairs/manifest.csv" \
#   --checkpoint "$CHECKPOINT" \
#   --out results/colab


## 7. Classical baseline comparison

In [ ]:
# The classical methods use the same test clips and the same supplied reference noise.
# This is a controlled algorithmic baseline, not a claim about a single-microphone system.
# !python scripts/run_baselines.py \
#   "$PROCESSED/eval_pairs/manifest.csv" \
#   --out results/colab/baselines.csv


## 8. Compare by method, SNR and noise type

In [ ]:
# !python scripts/plot_results.py \
#   --baseline results/colab/baselines.csv \
#   --hybrid results/colab/hybrid_fullband.csv \
#   --out results/colab/plots


In [ ]:
from pathlib import Path
import pandas as pd
for f in ['results/colab/baselines.csv','results/colab/hybrid_fullband.csv','results/colab/hybrid_by_noise_type.csv','results/colab/hybrid_by_snr.csv']:
    p=Path(f)
    if p.exists():
        print('
###',f); display(pd.read_csv(p).head(20))


## 9. Unseen-noise evaluation

For a defensible unseen-noise claim, create a separate noise directory whose sources/categories were not used during training. Generate a new manifest from that directory and run the exact same evaluation. Do not mix the unseen noise into the training manifest.

In [ ]:
# Example after preparing a separate unseen-noise directory:
# !python scripts/prepare_dataset.py \
#   --speech data/raw/unseen_test_speech \
#   --noise data/raw/unseen_noise \
#   --out data/processed/unseen \
#   --seed 999
# !python scripts/prepare_demo_manifest.py \
#   --speech-csv data/processed/unseen/test_speech.csv \
#   --noise-csv data/processed/unseen/noise.csv \
#   --out data/processed/unseen/eval_pairs \
#   --snrs -5,0,5,10,15 \
#   --seed 999
# !python scripts/evaluate_full_pipeline.py \
#   data/processed/unseen/eval_pairs/manifest.csv \
#   --checkpoint "$CHECKPOINT" \
#   --out results/unseen_noise


## 10. FP32 / INT8 and ONNX verification

In [ ]:
# Verify the repository's supplied ONNX model(s).
!python scripts/verify_onnx.py models/ampora_lite_dccrn_fp32.onnx
!python scripts/verify_onnx.py models/ampora_lite_dccrn_int8.onnx


## 11. Latency benchmark

In [ ]:
# Run on the actual Colab runtime and record the result. This is a machine-specific measurement.
!python scripts/benchmark_latency.py models/ampora_lite_dccrn_fp32.onnx
!python scripts/benchmark_latency.py models/ampora_lite_dccrn_int8.onnx


## 12. Optional: export a trained PyTorch checkpoint to ONNX

Only do this after the trained model is validated. The exact ONNX input shape depends on the selected STFT high-band dimensions.

In [ ]:
# Example export scaffold; run after confirming the checkpoint and input shape.
# import torch
# from src.models.dccrn import AMPORA_DCCRN
# model=AMPORA_DCCRN(16,64).eval()
# ck=torch.load(CHECKPOINT,map_location='cpu'); model.load_state_dict(ck.get('model',ck))
# dummy=torch.randn(1,2,204,95)  # replace 204/95 with the actual traced input shape
# torch.onnx.export(model,dummy,'models/colab_final/ampora_dccrn_fp32.onnx',opset_version=17,
#                   input_names=['stft_real_imag'],output_names=['complex_residual_mask'])


## 13. Final evidence table

Only copy values from generated CSVs into the PPT. Keep three labels separate: **Measured**, **Target**, and **Not yet measured**.

In [ ]:
from pathlib import Path
import pandas as pd
files = [
    'results/colab/baselines.csv',
    'results/colab/hybrid_fullband.csv',
    'results/unseen_noise/hybrid_fullband.csv',
]
for f in files:
    if Path(f).exists():
        df=pd.read_csv(f)
        cols=[c for c in ['Method','Noise_Type','SNR_dB','Delta_SI_SDR_dB','Delta_SNR_dB','Noisy_STOI','Enhanced_STOI','Noisy_PESQ','Enhanced_PESQ'] if c in df.columns]
        print('
',f); display(df[cols].groupby([c for c in ['Method','Noise_Type'] if c in cols]).mean(numeric_only=True).round(4))


## 14. What counts as a completed project

- [ ] Leakage-resistant train/validation/test split verified
- [ ] Controlled SNR mixing generated
- [ ] DC-CRN smoke test passes
- [ ] Final DC-CRN training completed
- [ ] Noisy vs DC-CRN metrics generated
- [ ] Spectral subtraction baseline generated
- [ ] Wiener baseline generated
- [ ] Reference-assisted NLMS baseline generated
- [ ] Software hybrid low-band NLMS + high-band DC-CRN evaluated
- [ ] Stationary/non-stationary/transient/mixed results generated
- [ ] Unseen-noise test generated
- [ ] FP32 vs INT8 verified/measured
- [ ] Latency measured on the intended runtime
- [ ] Plots generated
- [ ] README updated using actual measured values

Do not mark an item complete merely because code exists; the corresponding experiment must have been run.

## Required training-loop integration

After every completed epoch, save the complete training state:

```python
save_checkpoint(model, optimizer, scheduler, epoch, best_metric,
                history, config, project_root=DRIVE_ROOT)
```

When starting after a runtime disconnect:

```python
state = load_latest_checkpoint(model, optimizer, scheduler,
                                project_root=DRIVE_ROOT, map_location=device)
start_epoch = 0 if state is None else state['epoch'] + 1
print('Starting/resuming at epoch:', start_epoch)
```

This preserves model weights, optimizer state, scheduler state, epoch number, history and configuration—not just the weights.
